In [1]:
# Install required libraries
!pip install pandas numpy scikit-learn nltk

In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
# Load training dataset
train_path = "/kaggle/input/datasets/organizations/kaggle/recipe-ingredients-dataset/train.json"

with open(train_path) as f:
    data = json.load(f)

# Convert to dataframe
df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (39774, 3)


,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."


In [4]:
df['ingredients_text'] = df['ingredients'].apply(lambda x: " ".join(x))
df[['ingredients','ingredients_text']].head()

,ingredients,ingredients_text
0,"[romaine lettuce, black olives, grape tomatoes...",romaine lettuce black olives grape tomatoes ga...
1,"[plain flour, ground pepper, salt, tomatoes, g...",plain flour ground pepper salt tomatoes ground...
2,"[eggs, pepper, salt, mayonaise, cooking oil, g...",eggs pepper salt mayonaise cooking oil green c...
3,"[water, vegetable oil, wheat, salt]",water vegetable oil wheat salt
4,"[black pepper, shallots, cornflour, cayenne pe...",black pepper shallots cornflour cayenne pepper...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# fit and transform ingredient text
X = tfidf.fit_transform(df['ingredients_text'])

print("TF-IDF Matrix Shape:", X.shape)

TF-IDF Matrix Shape: (39774, 2970)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# compute cosine similarity matrix
cosine_sim = cosine_similarity(X, X)

print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (39774, 39774)


In [7]:
def recommend_recipes(ingredients_input, top_n=5):
    
    # convert input to TF-IDF vector
    input_vec = tfidf.transform([ingredients_input])
    
    # compute similarity with all recipes
    similarity_scores = cosine_similarity(input_vec, X).flatten()
    
    # get top recipe indices
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    # return top recipes
    results = df.iloc[top_indices][['cuisine', 'ingredients']]
    
    return results

In [8]:
recommend_recipes("egg rice onion")

,cuisine,ingredients
19547,vietnamese,"[water, oil, white rice, fine egg noodles, salt]"
32846,brazilian,"[egg yolks, shredded coconut, white sugar, but..."
7998,french,"[flour, sugar, butter, egg whites, baking choc..."
4462,brazilian,"[egg whites, shredded coconut, butter, granula..."
24339,french,"[butter, sugar, salt, whipping cream, egg yolk..."


In [9]:
# convert ingredients list to lowercase string for filtering
df["ingredients_joined"] = df["ingredients"].apply(lambda x: " ".join(i.lower() for i in x))

df[['ingredients', 'ingredients_joined']].head()

,ingredients,ingredients_joined
0,"[romaine lettuce, black olives, grape tomatoes...",romaine lettuce black olives grape tomatoes ga...
1,"[plain flour, ground pepper, salt, tomatoes, g...",plain flour ground pepper salt tomatoes ground...
2,"[eggs, pepper, salt, mayonaise, cooking oil, g...",eggs pepper salt mayonaise cooking oil green c...
3,"[water, vegetable oil, wheat, salt]",water vegetable oil wheat salt
4,"[black pepper, shallots, cornflour, cayenne pe...",black pepper shallots cornflour cayenne pepper...


In [10]:
def filter_allergies(dataframe, allergy_list):
    
    if not allergy_list:
        return dataframe
    
    filtered_df = dataframe.copy()
    
    for allergy in allergy_list:
        filtered_df = filtered_df[~filtered_df['ingredients_joined'].str.contains(allergy)]
    
    return filtered_df

In [11]:
def recommend_recipes_advanced(ingredients_input, allergies=None, cuisine=None, top_n=5):
    
    if allergies is None:
        allergies = []
    
    # filter allergies
    filtered_df = filter_allergies(df, allergies)
    
    # filter cuisine if provided
    if cuisine:
        filtered_df = filtered_df[filtered_df['cuisine'] == cuisine]
    
    # TF-IDF vector for input
    input_vec = tfidf.transform([ingredients_input])
    
    # similarity with all recipes
    similarity_scores = cosine_similarity(input_vec, X).flatten()
    
    filtered_indices = filtered_df.index
    
    similarity_scores = [(i, similarity_scores[i]) for i in filtered_indices]
    
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    top_indices = [i[0] for i in similarity_scores[:top_n]]
    
    return df.loc[top_indices][['cuisine','ingredients']]

In [12]:
recommend_recipes_advanced(
    ingredients_input="egg rice onion",
    allergies=["peanut"],
    cuisine="indian",
    top_n=5
)

,cuisine,ingredients
5942,indian,"[rice, salt, coconut, water, oil]"
9947,indian,"[kosher salt, rice flour, water]"
4010,indian,"[steamed rice, sugar, salt, dry yeast, coconut..."
12327,indian,"[salt, sugar, coconut milk, cooked rice, rice,..."
7901,indian,"[fresh cilantro, purple onion, ground coriande..."


In [13]:
# diet goal detection rules

high_protein_items = ["egg", "chicken", "beans", "lentils", "tofu", "paneer"]
vegan_items = ["tofu", "soy", "beans", "lentils"]
meat_items = ["chicken", "beef", "pork", "fish", "mutton"]

def detect_diet_type(ingredients_text):
    
    ingredients_text = ingredients_text.lower()
    
    if any(item in ingredients_text for item in meat_items):
        return "non_vegetarian"
    
    if any(item in ingredients_text for item in vegan_items):
        return "vegan"
    
    if any(item in ingredients_text for item in high_protein_items):
        return "high_protein"
    
    return "vegetarian"


df["diet_type"] = df["ingredients_joined"].apply(detect_diet_type)

df[["ingredients_joined","diet_type"]].head()

,ingredients_joined,diet_type
0,romaine lettuce black olives grape tomatoes ga...,vegan
1,plain flour ground pepper salt tomatoes ground...,high_protein
2,eggs pepper salt mayonaise cooking oil green c...,non_vegetarian
3,water vegetable oil wheat salt,vegetarian
4,black pepper shallots cornflour cayenne pepper...,non_vegetarian


In [14]:
def recommend_recipes_final(ingredients_input,
                            allergies=None,
                            cuisine=None,
                            diet_goal=None,
                            top_n=5):
    
    if allergies is None:
        allergies = []
    
    filtered_df = df.copy()
    
    # allergy filtering
    for allergy in allergies:
        filtered_df = filtered_df[
            ~filtered_df["ingredients_joined"].str.contains(allergy)
        ]
    
    # cuisine filtering
    if cuisine:
        filtered_df = filtered_df[
            filtered_df["cuisine"] == cuisine
        ]
    
    # diet goal filtering
    if diet_goal:
        filtered_df = filtered_df[
            filtered_df["diet_type"] == diet_goal
        ]
    
    # TF-IDF vector
    input_vec = tfidf.transform([ingredients_input])
    
    # similarity scores
    similarity_scores = cosine_similarity(input_vec, X).flatten()
    
    # keep only filtered indices
    filtered_indices = filtered_df.index
    
    similarity_scores = [(i, similarity_scores[i]) for i in filtered_indices]
    
    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)
    
    top_indices = [i[0] for i in similarity_scores[:top_n]]
    
    return df.loc[top_indices][["cuisine","diet_type","ingredients"]]

In [15]:
recommend_recipes_final(
    ingredients_input="egg rice onion",
    allergies=["peanut"],
    cuisine="indian",
    diet_goal="high_protein",
    top_n=5
)

,cuisine,diet_type,ingredients
23651,indian,high_protein,"[granulated sugar, large egg yolks, heavy crea..."
37900,indian,high_protein,"[pepper, extra-virgin olive oil, black mustard..."
7374,indian,high_protein,"[curry powder, salt, ground red pepper, egg wh..."
17084,indian,high_protein,"[garam masala, garlic cloves, tumeric, salt, o..."
21264,indian,high_protein,"[curry powder, ground black pepper, salt, sea ..."


In [16]:
# simple evaluation: check similarity score for sample inputs

def evaluate_model(test_inputs):
    
    for query in test_inputs:
        
        input_vec = tfidf.transform([query])
        similarity_scores = cosine_similarity(input_vec, X).flatten()
        
        top_index = similarity_scores.argmax()
        
        print("Query:", query)
        print("Top predicted cuisine:", df.iloc[top_index]["cuisine"])
        print("Ingredients:", df.iloc[top_index]["ingredients"])
        print("-"*60)


test_queries = [
    "chicken garlic onion",
    "rice coconut curry",
    "egg milk sugar",
    "beans tomato onion"
]

evaluate_model(test_queries)

Query: chicken garlic onion
Top predicted cuisine: indian
Ingredients: ['chicken', 'chicken breasts']
------------------------------------------------------------
Query: rice coconut curry
Top predicted cuisine: indian
Ingredients: ['rice', 'salt', 'coconut', 'water', 'oil']
------------------------------------------------------------
Query: egg milk sugar
Top predicted cuisine: french
Ingredients: ['sugar', 'egg yolks', 'milk']
------------------------------------------------------------
Query: beans tomato onion
Top predicted cuisine: greek
Ingredients: ['pepper', 'diced tomatoes', 'green beans', 'olive oil', 'salt', 'water', 'garlic', 'beans', 'parsley', 'yellow onion']
------------------------------------------------------------


In [17]:
import pickle

# save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# save TF-IDF feature matrix
with open("recipe_features.pkl", "wb") as f:
    pickle.dump(X, f)

# save processed dataframe
with open("recipes_dataframe.pkl", "wb") as f:
    pickle.dump(df, f)

print("Model files saved successfully!")

Model files saved successfully!


In [18]:
import os

os.listdir()

['.virtual_documents',
 'recipes_dataframe.pkl',
 'recipe_features.pkl',
 'tfidf_vectorizer.pkl']

In [19]:
# ============================================
# Personalized Recipe Recommendation Pipeline
# ============================================

# Step 1: Recipe Dataset
# Load recipe dataset (train.json) containing:
# id, cuisine, ingredients

# Step 2: Data Cleaning
# Convert JSON data to pandas DataFrame
# Remove unwanted formatting if needed

# Step 3: Ingredient Text Processing
# Convert ingredient list -> single text string
# Example:
# ['egg', 'rice', 'salt'] -> "egg rice salt"

# Step 4: TF-IDF Feature Extraction
# Use TfidfVectorizer to convert ingredient text
# into numerical feature vectors

# Step 5: Cosine Similarity Model
# Compute cosine similarity between recipe vectors
# This helps find recipes with similar ingredients

# Step 6: Allergy Filtering
# Remove recipes containing ingredients that
# the user is allergic to (example: peanut, milk)

# Step 7: Diet Goal Detection
# Detect diet types from ingredients
# Example:
# egg/chicken -> high_protein
# tofu/beans -> vegan
# meat -> non_vegetarian

# Step 8: Cuisine Filtering
# Allow user to filter recipes based on cuisine
# Example: Indian, Chinese, Italian

# Step 9: Recipe Recommendation
# Input:
#   ingredients + allergies + diet_goal + cuisine
# Output:
#   Top N recommended recipes

# Step 10: Save Model
# Save trained objects using pickle:
#   tfidf_vectorizer.pkl
#   recipe_features.pkl
#   recipes_dataframe.pkl

print("Pipeline documentation cell executed successfully.")

Pipeline documentation cell executed successfully.
